# RE-RL: Полный Pipeline для Formal Math (Lean 4 + Mathlib4)

Этот notebook — **единая точка входа** для генерации данных для theorem proving.

## Два источника данных

| Источник | Описание | Когда использовать |
|----------|----------|--------------------|
| **Mathlib Extraction** | Готовые доказательства из Mathlib | Базовый датасет, быстро |
| **BFS LeanNavigator** | Новые пути доказательств через BFS | Data augmentation, exploration |

## Pipeline

```
┌─────────────────────────────────────────────────────────────────┐
│  ШАГ 1: Трейсинг Mathlib4                                       │
│         fast_trace.py → clone, build, ExtractData → .ast.json   │
└───────────────────────────────┬─────────────────────────────────┘
                                │
            ┌───────────────────┴───────────────────┐
            ▼                                       ▼
┌───────────────────────────┐         ┌───────────────────────────┐
│  ПУТЬ A: Mathlib Extract  │         │  ПУТЬ B: BFS Navigator    │
│  (готовые доказательства) │         │  (новые пути)             │
│                           │         │                           │
│  extract_mathlib_proofs   │         │  ШАГ 2: Шаблоны тактик    │
│           │               │         │  ШАГ 3: RAG index         │
│           │               │         │  ШАГ 4: (Опц) Train BERT  │
│           │               │         │  ШАГ 5: BFS exploration   │
│           │               │         │           │               │
│           ▼               │         │           ▼               │
│    training pairs         │         │    training pairs         │
└───────────────────────────┘         └───────────────────────────┘
            │                                       │
            └───────────────────┬───────────────────┘
                                ▼
┌─────────────────────────────────────────────────────────────────┐
│  ШАГ 6: Объединение + Сохранение датасета                       │
│         JSONL / SFT / Chat format                               │
└─────────────────────────────────────────────────────────────────┘
```

## Скрипты в этой директории

| Файл | Описание |
|------|----------|
| `run_bfs.py` | BFS exploration (последовательный) |
| `run_bfs_parallel.py` | BFS exploration (параллельный, Ray) |
| `train_rag.py` | Обучение BERT RAG на (state, tactic) парах |
| `extract_mathlib_proofs.py` | Извлечение готовых доказательств из Mathlib |

---
## 0. Конфигурация

In [ ]:
# ══════════════════════════════════════════════════════════════
# КОНФИГУРАЦИЯ — измените под свои нужды
# ══════════════════════════════════════════════════════════════

MATHLIB_VERSION = "v4.26.0"  # https://github.com/leanprover-community/mathlib4/tags

# ── Источники данных ──
USE_MATHLIB_EXTRACT = True    # Извлечь готовые доказательства
USE_BFS_NAVIGATOR = True      # Генерировать новые пути через BFS
TRAIN_CUSTOM_RAG = False      # Обучить кастомный BERT RAG (требует много данных)

# ── Параметры извлечения из Mathlib ──
EXTRACT_MAX_THEOREMS = 50000  # 0 = все (медленно)
EXTRACT_MIN_PROOF_LENGTH = 2  # Минимум тактик в доказательстве
EXTRACT_MODULE_PREFIX = ""    # Фильтр (e.g., "Mathlib.Algebra")

# ── Параметры BFS ──
BFS_MAX_THEOREMS = 100        # Сколько теорем исследовать
BFS_MAX_STEPS = 10000         # Шагов BFS на теорему
BFS_MAX_TIME = 120            # Секунд на теорему
BFS_NO_AUTO = False           # Запретить simp/aesop (для длинных путей)
BFS_DECOMPOSE_AUTO = True     # Раскладывать simp на отдельные шаги
BFS_MIN_PROOF_LENGTH = 0      # Фильтр пар по distance_to_proof

# ── RAG ──
MIN_TEMPLATE_FREQ = 3         # Минимальная частота шаблона
RAG_MODEL = "sbert"           # "sbert" (pretrained) или "trained" (кастомный BERT)

# ── Вывод ──
OUTPUT_DIR = "../../datasets/formal_math_combined"
OUTPUT_FORMAT = "jsonl"       # jsonl | json | sft | chat

# ── Пути (автоматические) ──
import os
from pathlib import Path

CACHE_BASE = Path.home() / ".cache" / "re_rl"
REPO_DIR = CACHE_BASE / f"mathlib4-{MATHLIB_VERSION}" / "mathlib4"
NAV_DATA = CACHE_BASE / f"mathlib4-{MATHLIB_VERSION}" / "navigator_data"

os.environ["PATH"] = str(Path.home() / ".elan" / "bin") + ":" + os.environ.get("PATH", "")

print(f"Mathlib4:          {MATHLIB_VERSION}")
print(f"Mathlib Extract:   {USE_MATHLIB_EXTRACT} (max={EXTRACT_MAX_THEOREMS}, min_len={EXTRACT_MIN_PROOF_LENGTH})")
print(f"BFS Navigator:     {USE_BFS_NAVIGATOR} (max={BFS_MAX_THEOREMS}, steps={BFS_MAX_STEPS})")
print(f"Custom RAG:        {TRAIN_CUSTOM_RAG}")
print(f"Output:            {OUTPUT_DIR} ({OUTPUT_FORMAT})")

## 0.1 Проверка зависимостей

In [ ]:
def check_dependencies():
    """Проверяет все зависимости."""
    print("ПРОВЕРКА ЗАВИСИМОСТЕЙ")
    print("=" * 50)
    ok = True

    # elan
    elan = Path.home() / ".elan" / "bin" / "elan"
    if elan.exists():
        print(f"  elan:                  OK")
    else:
        print(f"  elan:                  НЕТ  →  curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh")
        ok = False

    # pantograph
    try:
        import pantograph
        print(f"  pantograph:            OK")
    except ImportError:
        print(f"  pantograph:            НЕТ  →  pip install pantograph")
        ok = False

    # faiss
    try:
        import faiss
        print(f"  faiss-cpu:             OK")
    except ImportError:
        print(f"  faiss-cpu:             НЕТ  →  pip install faiss-cpu")
        ok = False

    # sentence-transformers
    try:
        from sentence_transformers import SentenceTransformer
        print(f"  sentence-transformers: OK")
    except ImportError:
        print(f"  sentence-transformers: НЕТ  →  pip install sentence-transformers")
        ok = False

    # re_rl
    try:
        import re_rl
        print(f"  re_rl:                 OK")
    except ImportError:
        print(f"  re_rl:                 НЕТ  →  pip install -e ../..")
        ok = False

    print()
    if ok:
        print("Все зависимости установлены!")
    else:
        print("Установите недостающие и перезапустите ячейку.")
    return ok

DEPS_OK = check_dependencies()

In [ ]:
# Раскомментируйте если нужно установить
# !pip install pantograph faiss-cpu sentence-transformers
# !pip install -e ../..  # re_rl

## 0.2 Импорты

In [ ]:
import sys
import json
import time
import random
import subprocess
from collections import defaultdict
from datetime import datetime
from typing import List, Dict, Any

# Добавляем корень проекта в PATH
PROJECT_ROOT = Path(".").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from re_rl.tasks.formal.lean_navigator import (
    TacticTemplateExtractor,
    TacticRAG,
    PantographDojo,
    LeanNavigatorExplorer,
    TrainingPair,
    load_theorems_from_env,
)

print("Импорты загружены")

---
## Шаг 1: Трейсинг Mathlib4

Скрипт `fast_trace.py` выполняет:
1. `git clone --depth 1` Mathlib4
2. `lake exe cache get` — скачивание предсобранных .olean
3. `lake build` — сборка
4. `ExtractData.lean` — извлечение тактик и посылок в `.ast.json`

**Все этапы кэшируются** — при повторном запуске мгновенно пропускаются.

In [ ]:
# Проверяем: трейсинг уже выполнен?
build_ir = REPO_DIR / ".lake" / "build" / "ir"
tracing_done = (REPO_DIR.parent / ".step_5_done").exists()

if tracing_done:
    n_ast = sum(1 for _ in build_ir.rglob("*.ast.json")) if build_ir.exists() else 0
    print(f"✓ Трейсинг выполнен! (ast.json: {n_ast})")
    print(f"  Данные: {REPO_DIR}")
else:
    print(f"✗ Трейсинг НЕ выполнен для {MATHLIB_VERSION}")
    print(f"  Запустите в терминале:")
    print(f"    python scripts/fast_trace.py --version {MATHLIB_VERSION}")

In [ ]:
# Запуск трейсинга (если ещё не выполнен)
# Занимает ~60 мин для v4.26.0 на 24-ядерной машине

if not tracing_done:
    script = PROJECT_ROOT / "scripts" / "fast_trace.py"
    !python {script} --version {MATHLIB_VERSION}
    
    tracing_done = (REPO_DIR.parent / ".step_5_done").exists()
    if tracing_done:
        print("\n✓ Трейсинг завершён!")
    else:
        print("\n✗ Трейсинг не завершился. Проверьте ошибки выше.")
else:
    print("Трейсинг уже выполнен — пропускаем.")

---
# ПУТЬ A: Извлечение готовых доказательств из Mathlib

**Быстро и просто** — напрямую извлекаем (state, tactic) пары из traced данных.

Даёт:
- Реальные человеческие доказательства
- Многошаговые proof sequences
- ~100k+ теорем за минуты

In [ ]:
mathlib_pairs = []
mathlib_theorems = []

if USE_MATHLIB_EXTRACT and tracing_done:
    print("="*60)
    print("  ПУТЬ A: Извлечение из Mathlib")
    print("="*60)
    
    # Импортируем функции извлечения
    # (можно также запустить extract_mathlib_proofs.py напрямую)
    from extract_mathlib_proofs import (
        extract_theorems_from_ast,
        theorems_to_pairs,
    )
    
    ast_files = sorted(build_ir.rglob("*.ast.json"))
    ast_files = [f for f in ast_files if "packages" not in str(f.relative_to(build_ir))]
    
    if EXTRACT_MODULE_PREFIX:
        prefix_path = EXTRACT_MODULE_PREFIX.replace(".", "/")
        ast_files = [f for f in ast_files if prefix_path in str(f.relative_to(build_ir))]
    
    print(f"AST файлов: {len(ast_files)}")
    print(f"Извлекаем теоремы (min_length={EXTRACT_MIN_PROOF_LENGTH})...")
    
    t0 = time.time()
    for ast_file in ast_files:
        rel = ast_file.relative_to(build_ir)
        module_name = str(rel).replace(".ast.json", "").replace("/", ".")
        lean_rel = Path(str(rel).replace(".ast.json", ".lean"))
        lean_file = REPO_DIR / lean_rel
        
        if not lean_file.exists():
            continue
        
        theorems = extract_theorems_from_ast(
            ast_file, lean_file, module_name,
            min_proof_length=EXTRACT_MIN_PROOF_LENGTH,
            max_proof_length=50,
        )
        mathlib_theorems.extend(theorems)
        
        if EXTRACT_MAX_THEOREMS > 0 and len(mathlib_theorems) >= EXTRACT_MAX_THEOREMS:
            mathlib_theorems = mathlib_theorems[:EXTRACT_MAX_THEOREMS]
            break
    
    mathlib_pairs = theorems_to_pairs(mathlib_theorems)
    elapsed = time.time() - t0
    
    print(f"\n✓ Извлечено: {len(mathlib_theorems)} теорем, {len(mathlib_pairs)} pairs")
    print(f"  Время: {elapsed:.1f}с")
    
    # Статистика длин
    lengths = [t["proof_length"] for t in mathlib_theorems]
    print(f"  Средняя длина: {sum(lengths)/len(lengths):.1f} тактик")
else:
    if not USE_MATHLIB_EXTRACT:
        print("Пропускаем извлечение из Mathlib (USE_MATHLIB_EXTRACT=False)")
    else:
        print("Трейсинг не выполнен — извлечение невозможно")

---
# ПУТЬ B: BFS LeanNavigator

**Генерация новых путей доказательств** через Breadth-First Search.

Преимущества перед извлечением:
- Находит **альтернативные** доказательства (которых нет в Mathlib)
- Исследует пространство состояний
- Data augmentation — из 1 теоремы → много training pairs
- Полезно для RL (exploration trajectories)

## Шаг 2: Извлечение шаблонов тактик

In [ ]:
extractor = None
rag = None

if USE_BFS_NAVIGATOR and tracing_done:
    templates_path = NAV_DATA / "tactic_templates.json"
    NAV_DATA.mkdir(parents=True, exist_ok=True)
    
    extractor = TacticTemplateExtractor()
    
    if templates_path.exists():
        extractor.load(str(templates_path))
        print("✓ Шаблоны загружены из кэша")
    else:
        print("Извлекаем шаблоны из ast.json...")
        t0 = time.time()
        extractor.extract_from_ast_dir(str(REPO_DIR))
        print(f"  Время: {time.time()-t0:.1f}с")
        extractor.save(str(templates_path))
    
    print(f"\nТоп-10 шаблонов:")
    for tmpl, freq in extractor.get_top_templates(10):
        print(f"  [{freq:6d}] {tmpl[:60]}")
else:
    print("Пропускаем (BFS не используется или трейсинг не выполнен)")

## Шаг 3: RAG index (SBERT)

In [ ]:
if USE_BFS_NAVIGATOR and extractor is not None:
    rag_path = NAV_DATA / "rag_index"
    
    rag = TacticRAG(model_name="all-MiniLM-L6-v2")
    
    if (rag_path / "faiss.index").exists():
        rag.load(str(rag_path))
        print("✓ RAG загружен из кэша")
    else:
        print("Строим FAISS index...")
        t0 = time.time()
        rag.build_index(extractor.templates, min_freq=MIN_TEMPLATE_FREQ)
        print(f"  Время: {time.time()-t0:.1f}с")
        rag.save(str(rag_path))
        print("✓ RAG сохранён")
else:
    print("Пропускаем")

## Шаг 4 (Опционально): Обучение кастомного BERT RAG

По умолчанию используется pretrained SBERT (`all-MiniLM-L6-v2`).

Для лучшего retrieval можно обучить BERT на (state, tactic) парах из Mathlib.
Это требует:
1. Много данных (>10k pairs)
2. GPU
3. ~30 мин обучения

In [ ]:
if TRAIN_CUSTOM_RAG and len(mathlib_pairs) > 1000:
    print("="*60)
    print("  Обучение кастомного BERT RAG")
    print("="*60)
    print(f"Training pairs: {len(mathlib_pairs)}")
    print()
    print("Запустите в терминале:")
    print(f"  python train_rag.py --data <путь_к_pairs.jsonl> --output {NAV_DATA}/trained_rag")
    print()
    print("Или раскомментируйте код ниже:")
elif TRAIN_CUSTOM_RAG:
    print(f"Недостаточно данных для обучения RAG ({len(mathlib_pairs)} pairs)")
    print("Нужно минимум 1000, рекомендуется 10000+")
else:
    print("Пропускаем обучение RAG (TRAIN_CUSTOM_RAG=False)")
    print("Используем pretrained SBERT")

In [ ]:
# # Обучение RAG (раскомментируйте)
# if TRAIN_CUSTOM_RAG and len(mathlib_pairs) > 1000:
#     from train_rag import train_tactic_rag
#     
#     # Сохраняем pairs для обучения
#     train_data_path = NAV_DATA / "train_pairs.jsonl"
#     with open(train_data_path, "w") as f:
#         for p in mathlib_pairs:
#             f.write(json.dumps(p, ensure_ascii=False) + "\n")
#     
#     trained_rag_path = NAV_DATA / "trained_rag"
#     train_tactic_rag(
#         data_path=str(train_data_path),
#         output_dir=str(trained_rag_path),
#         epochs=3,
#         batch_size=32,
#     )
#     
#     # Используем обученный RAG
#     from re_rl.tasks.formal.lean_navigator import TrainedTacticRAG
#     rag = TrainedTacticRAG()
#     rag.load(str(trained_rag_path))
#     RAG_MODEL = "trained"
#     print("✓ Обученный RAG загружен")

## Шаг 5: BFS Exploration

In [ ]:
bfs_pairs = []
bfs_results = []

if USE_BFS_NAVIGATOR and rag is not None and tracing_done:
    print("="*60)
    print("  ПУТЬ B: BFS LeanNavigator")
    print("="*60)
    print(f"Параметры:")
    print(f"  max_theorems:  {BFS_MAX_THEOREMS}")
    print(f"  max_steps:     {BFS_MAX_STEPS}")
    print(f"  max_time:      {BFS_MAX_TIME}s")
    print(f"  no_auto:       {BFS_NO_AUTO}")
    print(f"  decompose:     {BFS_DECOMPOSE_AUTO}")
    print()
    
    # Banned tactics
    banned_tactics = None
    if BFS_NO_AUTO:
        banned_tactics = {
            "simp", "simp_all", "simpa", "simp_arith",
            "aesop", "decide", "trivial", "tauto",
            "omega", "linarith", "nlinarith", "positivity",
            "norm_num", "norm_cast", "push_cast",
            "ring", "ring_nf", "field_simp",
        }
        print(f"  Забанены: {sorted(banned_tactics)}")
        print()
    
    total_start = time.time()
    
    with PantographDojo(project_path=str(REPO_DIR), imports=["Mathlib"]) as dojo:
        # Загружаем теоремы из Lean env
        theorems = load_theorems_from_env(
            dojo,
            module_prefix="Mathlib",
            max_theorems=BFS_MAX_THEOREMS,
            cache_dir=str(NAV_DATA),
        )
        
        print(f"\nЗагружено {len(theorems)} теорем")
        print(f"Запускаем BFS...\n")
        
        explorer = LeanNavigatorExplorer(
            dojo=dojo,
            rag=rag,
            max_steps=BFS_MAX_STEPS,
            max_time=BFS_MAX_TIME,
            banned_tactics=banned_tactics,
            decompose_auto=BFS_DECOMPOSE_AUTO,
            verbose=False,
        )
        
        for i, thm in enumerate(theorems):
            t0 = time.time()
            try:
                result = explorer.explore(
                    goal_expr=thm.goal_expr,
                    theorem_name=thm.name,
                    theorem_code=thm.goal_state,
                    exit_on_finish=False,
                )
                
                # Фильтр по min_proof_length
                pairs = result.pairs
                if BFS_MIN_PROOF_LENGTH > 0:
                    pairs = [p for p in pairs if p.distance_to_proof >= BFS_MIN_PROOF_LENGTH]
                
                bfs_pairs.extend(pairs)
                elapsed = time.time() - t0
                
                bfs_results.append({
                    "theorem": thm.name,
                    "proven": result.theorem_proven,
                    "states": result.n_states,
                    "pairs": len(pairs),
                    "time": elapsed,
                })
                
                if (i + 1) % 10 == 0 or result.theorem_proven:
                    proven_cnt = sum(1 for r in bfs_results if r["proven"])
                    total_pairs = sum(r["pairs"] for r in bfs_results)
                    status = "PROVEN" if result.theorem_proven else "------"
                    print(f"  [{i+1:4d}/{len(theorems)}] {status} | "
                          f"proven={proven_cnt} pairs={total_pairs} | "
                          f"{thm.name[:40]} ({elapsed:.1f}s)")
                    
            except Exception as e:
                bfs_results.append({
                    "theorem": thm.name,
                    "proven": False,
                    "error": str(e)[:80],
                    "pairs": 0,
                })
    
    total_time = time.time() - total_start
    proven_cnt = sum(1 for r in bfs_results if r.get("proven"))
    
    print(f"\n✓ BFS завершён")
    print(f"  Теорем: {len(bfs_results)}, доказано: {proven_cnt}")
    print(f"  Training pairs: {len(bfs_pairs)}")
    print(f"  Время: {total_time:.1f}с")
else:
    print("Пропускаем BFS")

---
## Шаг 6: Объединение и сохранение датасета

In [ ]:
print("="*60)
print("  ИТОГОВАЯ СТАТИСТИКА")
print("="*60)

print(f"\nИсточник A (Mathlib Extraction):")
print(f"  Теорем:        {len(mathlib_theorems)}")
print(f"  Training pairs: {len(mathlib_pairs)}")

print(f"\nИсточник B (BFS Navigator):")
print(f"  Теорем:        {len(bfs_results)}")
print(f"  Training pairs: {len(bfs_pairs)}")

# Объединяем
all_pairs = []

# Mathlib pairs
for p in mathlib_pairs:
    all_pairs.append({
        "state": p["state"],
        "tactic": p["tactic"],
        "theorem_name": p["theorem_name"],
        "distance_to_proof": p["distance_to_proof"],
        "source": "mathlib",
    })

# BFS pairs
for p in bfs_pairs:
    all_pairs.append({
        "state": p.state,
        "tactic": p.tactic,
        "theorem_name": p.theorem_name,
        "distance_to_proof": p.distance_to_proof,
        "next_state": p.next_state,
        "source": "bfs",
    })

print(f"\nВСЕГО: {len(all_pairs)} training pairs")

In [ ]:
# Анализ тактик
if all_pairs:
    tactic_counts = defaultdict(int)
    for p in all_pairs:
        base_tac = p["tactic"].split()[0] if p["tactic"] else ""
        tactic_counts[base_tac] += 1
    
    print(f"\nТоп-15 тактик:")
    for tac, cnt in sorted(tactic_counts.items(), key=lambda x: -x[1])[:15]:
        pct = 100 * cnt / len(all_pairs)
        print(f"  {tac:25s} {cnt:6d} ({pct:.1f}%)")
    
    # Распределение по distance
    dist_counts = defaultdict(int)
    for p in all_pairs:
        dist_counts[p["distance_to_proof"]] += 1
    
    print(f"\nРаспределение по distance_to_proof:")
    for d in sorted(dist_counts.keys())[:10]:
        print(f"  distance={d}: {dist_counts[d]} pairs")

In [ ]:
def save_combined_dataset(pairs, output_dir, fmt, metadata):
    """Сохраняет объединённый датасет."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if fmt == "jsonl":
        f = output_dir / f"formal_math_{ts}.jsonl"
        with open(f, "w") as fh:
            for p in pairs:
                fh.write(json.dumps(p, ensure_ascii=False) + "\n")
    elif fmt == "json":
        f = output_dir / f"formal_math_{ts}.json"
        with open(f, "w") as fh:
            json.dump(pairs, fh, indent=2, ensure_ascii=False)
    elif fmt == "sft":
        f = output_dir / f"formal_math_sft_{ts}.json"
        sft = [{
            "instruction": "You are a Lean 4 theorem prover. Given the current proof state, output the next tactic.",
            "input": f"Proof state:\n{p['state']}",
            "output": p["tactic"],
        } for p in pairs if p["tactic"]]
        with open(f, "w") as fh:
            json.dump(sft, fh, indent=2, ensure_ascii=False)
    elif fmt == "chat":
        f = output_dir / f"formal_math_chat_{ts}.json"
        chat = [{
            "messages": [
                {"role": "system", "content": "You are an expert Lean 4 theorem prover."},
                {"role": "user", "content": f"Current proof state:\n```\n{p['state']}\n```\nWhat tactic should I apply?"},
                {"role": "assistant", "content": p["tactic"]},
            ]
        } for p in pairs if p["tactic"]]
        with open(f, "w") as fh:
            json.dump(chat, fh, indent=2, ensure_ascii=False)
    else:
        raise ValueError(f"Unknown format: {fmt}")
    
    # Метаданные
    mf = output_dir / f"metadata_{ts}.json"
    with open(mf, "w") as fh:
        json.dump(metadata, fh, indent=2, ensure_ascii=False, default=str)
    
    print(f"\nСохранено:")
    print(f"  Датасет:    {f}  ({f.stat().st_size:,} bytes)")
    print(f"  Метаданные: {mf}")
    return f


if all_pairs:
    metadata = {
        "mathlib_version": MATHLIB_VERSION,
        "generation_date": datetime.now().isoformat(),
        "sources": {
            "mathlib_extraction": {
                "enabled": USE_MATHLIB_EXTRACT,
                "theorems": len(mathlib_theorems),
                "pairs": len(mathlib_pairs),
            },
            "bfs_navigator": {
                "enabled": USE_BFS_NAVIGATOR,
                "theorems": len(bfs_results),
                "pairs": len(bfs_pairs),
                "proven": sum(1 for r in bfs_results if r.get("proven")),
            },
        },
        "config": {
            "extract_max_theorems": EXTRACT_MAX_THEOREMS,
            "extract_min_proof_length": EXTRACT_MIN_PROOF_LENGTH,
            "bfs_max_theorems": BFS_MAX_THEOREMS,
            "bfs_max_steps": BFS_MAX_STEPS,
            "bfs_no_auto": BFS_NO_AUTO,
            "bfs_decompose_auto": BFS_DECOMPOSE_AUTO,
        },
        "total_pairs": len(all_pairs),
    }
    
    output_file = save_combined_dataset(all_pairs, OUTPUT_DIR, OUTPUT_FORMAT, metadata)
else:
    print("Нет данных для сохранения.")

---
## Быстрый тест: BFS на простых теоремах

In [ ]:
# Быстрый тест на конкретных теоремах
if rag is not None:
    test_goals = [
        ("Nat.add_comm", "∀ (n m : ℕ), n + m = m + n"),
        ("Nat.zero_le",  "∀ (n : ℕ), 0 ≤ n"),
        ("Int.add_comm", "∀ (a b : ℤ), a + b = b + a"),
    ]
    
    print("Быстрый тест BFS:")
    print()
    
    with PantographDojo(project_path=str(REPO_DIR), imports=["Mathlib"]) as dojo:
        quick_explorer = LeanNavigatorExplorer(
            dojo=dojo, rag=rag,
            max_steps=3000, max_time=15, verbose=True,
        )
        for name, goal in test_goals:
            print(f"{'='*50}")
            print(f"[BFS] {name}: {goal}")
            result = quick_explorer.explore(
                goal_expr=goal, theorem_name=name,
                theorem_code=goal, exit_on_finish=True,
            )
            s = "✓ PROVEN" if result.theorem_proven else "✗ not proven"
            print(f"  {s} | states={result.n_states} pairs={len(result.pairs)}")
else:
    print("RAG не инициализирован — пропускаем тест")

---
## Альтернатива: Запуск через командную строку

Для масштабной генерации удобнее использовать скрипты напрямую:

```bash
# 1. Трейсинг (один раз)
python scripts/fast_trace.py --version v4.26.0

# 2a. Извлечение из Mathlib (быстро)
python examples/formal/extract_mathlib_proofs.py \
    --min-proof-length 2 \
    --max-theorems 50000

# 2b. BFS exploration (медленно, но новые данные)
python examples/formal/run_bfs.py \
    --max-theorems 1000 \
    --max-steps 50000 \
    --max-time 300 \
    --decompose-auto

# 2c. BFS параллельно (Ray, быстрее)
python examples/formal/run_bfs_parallel.py \
    --max-theorems 1000 \
    --workers 8 \
    --decompose-auto

# 3. (Опционально) Обучение RAG
python examples/formal/train_rag.py \
    --data datasets/formal_math_combined/*.jsonl \
    --output ~/.cache/re_rl/mathlib4-v4.26.0/navigator_data/trained_rag
```

---
## Справка

### Структура кэша
```
~/.cache/re_rl/mathlib4-v4.26.0/
  ├── .step_1_done ... .step_5_done   # маркеры этапов
  ├── mathlib4/                        # исходники + build
  │   ├── Mathlib/                     # .lean файлы
  │   └── .lake/build/ir/              # .ast.json
  └── navigator_data/                  # кэш для BFS
      ├── tactic_templates.json        # шаблоны
      ├── rag_index/                   # FAISS index
      └── trained_rag/                 # (опционально) обученный BERT
```

### Форматы датасета

| Формат | Описание | Использование |
|--------|----------|---------------|
| `jsonl` | JSON Lines | Streaming, большие датасеты |
| `json` | JSON array | Небольшие датасеты |
| `sft` | Instruction/Input/Output | Supervised fine-tuning |
| `chat` | Messages array | Chat fine-tuning |

### Масштабирование

| Режим | Mathlib Extract | BFS | Время | Результат |
|-------|-----------------|-----|-------|----------|
| Тест | 1000 теорем | 10 теорем | 2 мин | ~5k pairs |
| Средний | 50000 теорем | 500 теорем | 2 часа | ~200k pairs |
| Полный | все | 10000 теорем | дни | миллионы |

In [ ]:
print("="*60)
print("  ГОТОВО!")
print("="*60)
print(f"\nСгенерировано {len(all_pairs)} training pairs")
print(f"  из Mathlib: {len(mathlib_pairs)}")
print(f"  из BFS:     {len(bfs_pairs)}")
if all_pairs:
    print(f"\nДатасет: {OUTPUT_DIR}")